# Interchange

> Conversion to and from the Hypergraph Interchange Format (HIF)

[HIF](https://github.com/pszufe/HIF-standard) is a JSON standard for exchanging higher-order networks between libraries e.g. [XGI](https://github.com/xgi-org/xgi) and [HyperNetX](https://github.com/pnnl/HyperNetX) (Coll et al., 2025, *Network Science* 13; [arXiv:2507.11520](https://arxiv.org/abs/2507.11520)). The JSON schema in `parag/data/hif_schema.json` is from the HIF-standard repository (MIT License).

In [ ]:
from parag.io import to_hif, from_hif, validate_hif, write_hif, read_hif

## Examples

In [ ]:
#| include: false
import json
import pandas as pd
import tempfile
import networkx as nx
import xgi

Round trip of the network communities (undirected), in both modes, and reading by XGI.

In [ ]:
G = nx.karate_club_graph()
nodes = (
    pd.Series({i: list(t) for i, t in enumerate(list(nx.community.girvan_newman(G))[3])})
    .explode()
    .to_frame("node id")
    .reset_index()
    .rename(columns={"index": "community id"})
    .astype({"node id": int})
    .loc[:, ["node id", "community id"]]
)
edges = pd.DataFrame(G.edges, columns=["source", "target"])
for mode in ["subsets", "pairwise"]:
    hif = to_hif(nodes, edges, "node id", "source", "target", "community id", mode=mode)
    assert validate_hif(hif)
    d = from_hif(json.loads(json.dumps(hif)))
    pd.testing.assert_frame_equal(d["nodes"], nodes, check_dtype=False)
    pd.testing.assert_frame_equal(d["edges"], edges, check_dtype=False)
    H = xgi.convert.from_hif_dict(hif)
    assert H.num_nodes == len(nodes)
    assert H.num_edges == (nodes["community id"].nunique() if mode == "subsets" else len(edges))

Directed pairwise edges (drug → side effect), and writing/reading a HIF file.

In [ ]:
edges = pd.read_csv("examples/inputs/ChSe-Decagon_monopharmacy.csv")
nodes = pd.concat(
    [
        edges.loc[:, ["# STITCH"]].drop_duplicates().rename(columns={"# STITCH": "node id"}).assign(subset="drug"),
        edges.loc[:, ["Individual Side Effect", "Side Effect Name"]]
        .drop_duplicates(subset=["Individual Side Effect"])
        .rename(columns={"Individual Side Effect": "node id"})
        .assign(subset="side effect"),
    ],
    axis=0,
    ignore_index=True,
)
hif = to_hif(nodes, edges, "node id", "# STITCH", "Individual Side Effect", "subset", mode="pairwise", directed=True)
with tempfile.TemporaryDirectory() as dir_path:
    d = from_hif(read_hif(write_hif(hif, f"{dir_path}/drugs.json")))
pd.testing.assert_frame_equal(d["nodes"], nodes, check_dtype=False)
pd.testing.assert_frame_equal(d["edges"], edges, check_dtype=False)
assert type(xgi.convert.from_hif_dict(hif)).__name__ == "DiHypergraph"

A HIF written by another library (XGI), with nodes in multiple hyperedges.

In [ ]:
hif = dict(xgi.convert.to_hif_dict(xgi.Hypergraph([[1, 2, 3], [3, 4], [4, 5, 6, 7]])))
try:
    from_hif(hif)
    raise AssertionError("expected a ValueError")
except ValueError:
    pass
d = from_hif(hif, subset_rule="largest")
assert d["nodes"]["subset"].tolist() == [0, 0, 0, 2, 2, 2, 2]
assert len(d["edges"]) == 3 + 1 + 6  # g: clique expansion
assert from_hif(hif, subset_rule="first")["nodes"]["subset"].tolist() == [0, 0, 0, 1, 2, 2, 2]

Invalid HIF.

In [ ]:
assert not validate_hif({"nodes": []}, errors="warn")